In [0]:
notepoint_rel = "hands-on/day-07/notebooks/02-Day7-Delta-Live-Tables.ipynb"
BASE_PATH = "abfss://atininput@sadbtrng19032026.dfs.core.windows.net/data/"
STUDENT_ID = "u25"  # Change to your assigned u01–u16
DAY5 = BASE_PATH + "/day5"
P_BASIC = DAY5 + "/delta/flight_summary_basic"
DAY7_PREFIX = f"{BASE_PATH}day07-{STUDENT_ID}"
try:
    spark.read.format("delta").load(P_BASIC).limit(1).collect()
    print("✓ Day 5 tables found (P_BASIC)")
except Exception as e:
    print(f"✗ Day 5 tables not found: {e}")
    print("Please complete Day 5 notebook 01 first.")

print("DAY7_PREFIX (for any sink paths you add):", DAY7_PREFIX)



## Day 7 — Item 20: Delta Live Tables (DLT)

This notebook is meant to run as a **Delta Live Tables pipeline** in the Databricks UI (not as ad-hoc cells on a normal cluster). It defines Python `@dlt.table` objects that read the Day 5 Delta table `P_BASIC`, build bronze → silver → gold, and apply a simple **expectation** on the gold table.

**Agenda:** root `README.md` — Day 7, item 20.

**Steps:** Workspace → Workflows → Delta Live Tables → Create pipeline → set **Notebook path** to this notebook → choose a **Target schema** (for example `day07_<your_id>`) → Start.


In [0]:
import dlt
from pyspark.sql.functions import col


@dlt.table(comment="Bronze: read Day 5 flight_summary_basic Delta path")
def bronze_flights():
    return spark.read.format("delta").load(P_BASIC)


@dlt.view()
def silver_flights():
    return dlt.read("bronze_flights").filter(col("count").isNotNull())


@dlt.table(comment="Gold with expectation: non-negative count")
@dlt.expect("count_non_negative", "count >= 0")
def gold_flights():
    return dlt.read("silver_flights")


### Notes

- If `import dlt` fails, you are not running inside a DLT pipeline context; create and start a pipeline as above.
- Adjust `@dlt.expect` or add more tables to match how you teach data quality in class.
